In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.svm import SVR

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"C:\Users\dorap\Downloads\inv\house-prices-advanced-regression-techniques\train.csv")

print(df.shape)
df.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
df = df.select_dtypes(include=['number'])

df.fillna(df.median(), inplace=True)

X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest R²:", rf_r2)

Random Forest R²: 0.8840221174017272


In [6]:
gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

gbr.fit(X_train, y_train)

gbr_pred = gbr.predict(X_test)

gbr_r2 = r2_score(y_test, gbr_pred)

print("Gradient Boosting R²:", gbr_r2)

Gradient Boosting R²: 0.8995325229035104


In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svr = SVR(
    kernel='rbf',
    C=100,
    gamma='scale'
)

svr.fit(X_train_scaled, y_train)

svr_pred = svr.predict(X_test_scaled)

svr_r2 = r2_score(y_test, svr_pred)

print("SVR R²:", svr_r2)

SVR R²: 0.06807791507082472


In [8]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X_scaled)

print("Original Features:", X.shape[1])
print("Reduced Features:", X_pca.shape[1])

Original Features: 37
Reduced Features: 27


In [9]:
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca,
    y,
    test_size=0.2,
    random_state=42
)

rf_pca = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_pca.fit(X_train_pca, y_train_pca)

pred_pca = rf_pca.predict(X_test_pca)

print("RF + PCA R²:",
      r2_score(y_test_pca, pred_pca))

RF + PCA R²: 0.8738214254796852


In [10]:
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [11]:
rf_scores = cross_val_score(
    rf,
    X,
    y,
    cv=kfold,
    scoring='r2'
)

print("RF Mean R²:",
      rf_scores.mean())

RF Mean R²: 0.832293897158357


In [12]:
gbr_scores = cross_val_score(
    gbr,
    X,
    y,
    cv=kfold,
    scoring='r2'
)

print("GBR Mean R²:",
      gbr_scores.mean())

GBR Mean R²: 0.8353992739464214


In [13]:
svr_scores = cross_val_score(
    svr,
    X_scaled,
    y,
    cv=kfold,
    scoring='r2'
)

print("SVR Mean R²:",
      svr_scores.mean())

SVR Mean R²: 0.04944877421091296


In [14]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

estimators = [
    ('rf', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )),
    
    ('gbr', GradientBoostingRegressor(
        random_state=42
    ))
]

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge()
)

stack.fit(X_train, y_train)

stack_pred = stack.predict(X_test)

stack_r2 = r2_score(
    y_test,
    stack_pred
)

print("Stacking R²:", stack_r2)

Stacking R²: 0.9018049412385847


In [15]:
results = pd.DataFrame({
    'Model':[
        'Random Forest',
        'Gradient Boosting',
        'SVR',
        'Stacking'
    ],

    'R2 Score':[
        rf_r2,
        gbr_r2,
        svr_r2,
        stack_r2
    ]
})

results.sort_values(
    by='R2 Score',
    ascending=False
)

,Model,R2 Score
3,Stacking,0.901805
1,Gradient Boosting,0.899533
0,Random Forest,0.884022
2,SVR,0.068078


In [16]:
importance = pd.DataFrame({
    'Feature':X.columns,
    'Importance':rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(10))

        Feature  Importance
4   OverallQual    0.567407
16    GrLivArea    0.126303
12  TotalBsmtSF    0.037300
14     2ndFlrSF    0.031816
9    BsmtFinSF1    0.031475
13     1stFlrSF    0.030958
3       LotArea    0.021245
6     YearBuilt    0.017692
27   GarageArea    0.017491
26   GarageCars    0.016289


In [21]:
y_log = np.log1p(y)
y_train = np.log1p(y_train)
gbr_pred = np.expm1(gbr_pred)